# Aprovisionamiento, I/O y Cerrojo de Infraestructura (Colab GPU)

Tras el colapso empírico de la CPU local al intentar resolver la complejidad geométrica del modelo semántico (Cuaderno 04), migramos la infraestructura a un clúster en la nube (Google Colab) con aceleración nativa por hardware (GPU).

Este bloque aprovisiona el contenedor efímero, monta los volúmenes persistentes de Google Drive para evitar la fuga de datos y prepara el entorno `transformers` de HuggingFace. El objetivo final es aprovechar el paralelismo masivo de los núcleos Tensor para ejecutar un *Fine-Tuning End-to-End* en tiempo asumible por la operativa BPO.

In [11]:
# CELDA 1: Inyección Nativa de Dependencias
# ATENCIÓN: Reinicia obligatoriamente la sesión de Colab tras la instalación.
!pip install -q transformers accelerate datasets fastparquet scipy scikit-learn

In [12]:
# CELDA 2: I/O Persistente y Cerrojo de Hardware (Colab)
import os
import torch
import warnings
from google.colab import drive

warnings.filterwarnings('default')

print("Montando almacenamiento persistente de Google Drive...")
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR'
DATA_PATH = f'{BASE_PATH}/data/gold/train_set_telco.parquet'

print("Auditoría de hardware gráfico en curso...")
assert torch.cuda.is_available(), "FALLO CRÍTICO: Entorno limitado a CPU. Cambia el tipo de entorno y activa la GPU T4."
gpu_name = torch.cuda.get_device_name(0)
print(f"Cerrojo superado. Acelerador CUDA en línea: {gpu_name}")

Montando almacenamiento persistente de Google Drive...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Auditoría de hardware gráfico en curso...
Cerrojo superado. Acelerador CUDA en línea: NVIDIA A100-SXM4-80GB


# Ingesta de Matriz Telco y Metadatos Semánticos (Full-Shot)

Con la GPU estabilizada, cargamos la matriz completa de entrenamiento Telco (`train_set_telco.parquet`). Al abandonar el pesado aprendizaje contrastivo en favor de la clasificación de secuencias tradicional (*Sequence Classification*), la memoria VRAM puede asimilar el 100% de la volumetría en lotes convencionales.

Se instancia un codificador estático (`LabelEncoder`) para abstraer las 56 colas operativas del negocio a enteros matemáticos, generando el diccionario maestro que utilizará la capa de clasificación lineal que añadiremos encima del *Transformer*.

In [13]:
# CELDA 3: Ingesta Defensiva y Extracción de Metadatos
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("Ingestando matriz de entrenamiento desde disco persistente...")
df_train = pd.read_parquet(DATA_PATH, engine='fastparquet')

print("Ejecutando purga y casteo defensivo en la serie de texto...")
# Bloqueo algorítmico contra celdas corruptas o nulos generados en la descompresión
df_train['full_text'] = df_train['full_text'].fillna("").astype(str)

print("Codificando la variable objetivo (Label Encoding)...")
le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['target_tripleta'])

# Extracción bidireccional estricta (Requisito fundacional de Hugging Face)
id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in enumerate(le.classes_)}
num_classes = len(le.classes_)

print(f"Volumen asimilado: {len(df_train)} tickets | Clases objetivo extraídas: {num_classes}")

Ingestando matriz de entrenamiento desde disco persistente...
Ejecutando purga y casteo defensivo en la serie de texto...
Codificando la variable objetivo (Label Encoding)...
Volumen asimilado: 12322 tickets | Clases objetivo extraídas: 56


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Motor de Telemetría BPO (Métricas Híbridas y Blindaje de Tipos)

Este módulo evalúa el rendimiento del algoritmo calculando tanto el rigor estadístico clásico (F1, Log-Loss) como las métricas de negocio puras.

**El Cortafuegos de Pasividad:** Mantenemos la exigencia matemática estricta que aniquiló al Random Forest. La `Tasa de Automatización` se calculará exclusivamente sobre aquellas predicciones probabilísticas que superen un umbral de seguridad del **0.85**. Su aislamiento responde al principio innegociable de desacoplar la función de pérdida matemática de las métricas reales que impactan en el SLA del cliente.

In [14]:
# CELDA 4: Motor de Métricas BPO
import torch
import numpy as np
from sklearn.metrics import f1_score, log_loss, cohen_kappa_score

def calcular_metricas_bpo(y_true, y_prob, umbral=0.85):
    """
    Evaluador central de métricas híbridas (Negocio + Data Science).
    Restricción técnica: y_prob DEBE ser un array plano de NumPy.
    """
    # Trampa de tipos estricta (Compatible con la actualización NumPy 2.0)
    if torch.is_tensor(y_prob):
        raise TypeError("Violación de aislamiento: El motor BPO no acepta tensores de VRAM. Transfiere a NumPy (.cpu().numpy()) antes de inyectar.")

    y_pred = np.argmax(y_prob, axis=1)
    confianzas = np.max(y_prob, axis=1)

    # Segmentación táctica en base a la tolerancia operativa de negocio
    automatizados_mask = confianzas >= umbral

    total_tickets = len(y_true)
    tickets_automatizados = np.sum(automatizados_mask)
    tasa_automatizacion = tickets_automatizados / total_tickets if total_tickets > 0 else 0.0

    # Precisión matemática calculada exclusivamente sobre la capa de automatización
    if tickets_automatizados > 0:
        precision_condicionada = np.mean(y_true[automatizados_mask] == y_pred[automatizados_mask])
    else:
        precision_condicionada = 0.0

    # Telemetría de validación científica (Rigor estadístico)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    # Blindaje dimensional contra omisión de clases en el particionado
    entropia = log_loss(y_true, y_prob, labels=range(num_classes))

    return {
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'kappa': kappa,
        'log_loss': entropia,
        'tasa_automatizacion': tasa_automatizacion,
        'precision_condicionada': precision_condicionada
    }

# Orquestador Neuronal K-Fold y Función de Coste Penalizada

A pesar del alto coste computacional, sometemos a la red neuronal al mismo rigor de validación cruzada (5 Folds) que a los modelos estadísticos para garantizar que no haya sobreajuste estructural.

Para prevenir el hundimiento del hiperplano ante el desbalanceo extremo del BPO, se diseña una Función de Coste Ponderada (aplicando un suavizado de raíz cuadrada inversa sobre las frecuencias de las 56 clases Telco). Sobrescribimos la clase base `Trainer` inyectando este tensor dinámico mediante `**kwargs` directamente en el cálculo del gradiente (CrossEntropyLoss). Simultáneamente, la matriz de texto se tokeniza en frío al formato subword nativo de RoBERTa para maximizar la ingesta en la GPU.

In [15]:
# CELDA 5: Orquestador K-Fold Nativo (Contingencia RoBERTa con Pesos Penalizados Suavizados)
import os
import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from scipy.special import softmax

folds_unicos = sorted(df_train['fold_id'].unique())
resultados_kfold = []

# Ajuste de persistencia nativa en Colab
CSV_BACKUP_PATH = f"{BASE_PATH}/data/resultados_kfold_roberta_telco.csv"
if os.path.exists(CSV_BACKUP_PATH):
    os.remove(CSV_BACKUP_PATH)

print("Iniciando transición de memoria a Apache Arrow...")
dataset_global = Dataset.from_pandas(df_train[['full_text', 'label', 'fold_id']])
del df_train
gc.collect()

print("Ejecutando tokenización masiva en frío (Truncamiento a 256 tokens)...")
modelo_id = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)

def tokenizar_lote(batch):
    # CORRECCIÓN 3: Aseguramos max_length=256 para no asfixiar el contexto del ticket
    return tokenizer(batch["full_text"], padding="max_length", truncation=True, max_length=256)

dataset_tokenizado = dataset_global.map(tokenizar_lote, batched=True, batch_size=1000)
dataset_tokenizado = dataset_tokenizado.remove_columns(["full_text"])
del dataset_global
gc.collect()

# Sobreescritura del orquestador estándar para inyectar la función de coste penalizada
class BPOWeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # CORRECCIÓN 2: Mutación no destructiva usando .get() en lugar de .pop()
        labels = inputs.get("labels")
        # Filtramos las etiquetas para evitar que Hugging Face calcule su propia pérdida interna
        inputs_filtrados = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs = model(**inputs_filtrados)
        logits = outputs.get("logits")

        if self.class_weights.device != logits.device:
            self.class_weights = self.class_weights.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

print("\nInicializando motor K-Fold sobre GPU de Colab (RoBERTa FP16 + SqRt Class Weights)...")

for fold in folds_unicos:
    print(f"\n--- Compilando Fold {fold}/{len(folds_unicos)} ---")

    train_dataset = dataset_tokenizado.filter(lambda x: x['fold_id'] != fold)
    val_dataset = dataset_tokenizado.filter(lambda x: x['fold_id'] == fold)

    # CORRECCIÓN 1: Data Leakage mitigado. Los pesos se calculan estrictamente sobre el Train del Fold actual
    clases_unicas, frecuencias = np.unique(train_dataset['label'], return_counts=True)
    pesos_crudos = 1.0 / np.sqrt(frecuencias)
    pesos_numpy = pesos_crudos * (len(clases_unicas) / np.sum(pesos_crudos))
    tensor_pesos = torch.tensor(pesos_numpy, dtype=torch.float32)

    model = AutoModelForSequenceClassification.from_pretrained(
        modelo_id,
        num_labels=num_classes,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )

    args = TrainingArguments(
        output_dir="/content/tmp_trainer",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=4,
        fp16=True,
        num_train_epochs=10,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        learning_rate=1e-4,
        warmup_steps=300,
        weight_decay=0.01,
        max_grad_norm=1.0
    )

    trainer = BPOWeightedTrainer(
        class_weights=tensor_pesos,
        model=model,
        args=args,
        train_dataset=train_dataset
    )

    print("Inyectando tensores, matriz de pesos y actualizando pesos en CUDA...")
    trainer.train()

    print("Transfiriendo partición de validación e infiriendo Logits crudos...")
    salida_prediccion = trainer.predict(val_dataset)
    logits = salida_prediccion.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    if torch.is_tensor(logits):
        logits = logits.cpu().numpy()

    print("Normalizando Logits mediante Softmax...")
    y_prob = softmax(logits, axis=1)
    y_true = np.array(val_dataset['label'])

    metricas_fold = calcular_metricas_bpo(y_true, y_prob)
    metricas_fold['fold'] = fold
    resultados_kfold.append(metricas_fold)

    print(f"Fold {fold} estabilizado. Tasa Automatización: {metricas_fold['tasa_automatizacion']:.2%}")

    del train_dataset
    del val_dataset
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()

# CORRECCIÓN 4: Persistencia bloqueada hasta el final del bucle para evitar CSVs corruptos
print("\nBake-Off estadístico finalizado con éxito. Consolidando matriz K-Fold en disco.")
df_final = pd.DataFrame(resultados_kfold)
df_final.to_csv(CSV_BACKUP_PATH, index=False)

print("\nVolcado automático de métricas (CSV) por pantalla para auditoría inmediata:")
print(df_final.to_string())

Iniciando transición de memoria a Apache Arrow...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Ejecutando tokenización masiva en frío (Truncamiento a 256 tokens)...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Step,Training Loss
50,16.086919
100,14.358597
150,12.340514
200,11.131919
250,10.873831
300,10.626626
350,10.208978
400,10.105518
450,9.985448
500,9.631758


Transfiriendo partición de validación e infiriendo Logits crudos...


Normalizando Logits mediante Softmax...
Fold 4 estabilizado. Tasa Automatización: 7.59%

Bake-Off estadístico finalizado con éxito. Consolidando matriz K-Fold en disco.

Volcado automático de métricas (CSV) por pantalla para auditoría inmediata:
   f1_macro  f1_weighted     kappa  log_loss  tasa_automatizacion  precision_condicionada  fold
0  0.431373     0.450779  0.430359  1.880219             0.064503                0.811321     0
1  0.418362     0.422593  0.396349  1.951199             0.041379                0.833333     1
2  0.439905     0.439929  0.414014  1.977110             0.046672                0.886957     2
3  0.435021     0.429490  0.406281  1.919345             0.016640                0.902439     3
4  0.452164     0.462613  0.437782  1.897235             0.075893                0.791444     4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Entrenamiento Maestro y Auditoría de Producción (Hold-Out Ciego)

Superada la validación cruzada y demostrada la estabilidad de la red, procedemos a la compilación del artefacto predictivo final para el vertical Telco.

Este orquestador despliega tres fases críticas:
1. **Entrenamiento Acelerado (GPU):** Ingesta ininterrumpida de la matriz de entrenamiento aprovechando paralelismo de tensores (FP16).
2. **Violación de Cuarentena (Test Set):** Por primera y única vez, se infiere probabilísticamente y de forma ciega sobre el 20% del dataset retenido (`test_set_telco.parquet`).
3. **Telemetría Pura y Veredicto de Negocio:** La red extrae las métricas BPO finales y persiste estáticamente el modelo para su despliegue en un endpoint (FastAPI).

### El Veredicto Definitivo
Bajo el exigente cortafuegos operativo del 0.85 de confianza:
* El Machine Learning Clásico (Random Forest) automatizaba un **0.00%** (Log-Loss de 1.88).
* El Modelo Semántico Congelado (Embeddings + LogReg) lograba automatizar un 8.7% pero se equivocaba el **65%** de las veces (Precisión 35%).
* **SITOR (RoBERTa Fine-Tuned)** retiene una automatización sólida del **8.63%** garantizando una alta Precisión Condicionada del **83.46%**, con un Log-Loss calibrado de **1.75**.

La red neuronal es la única arquitectura matemáticamente capaz de calibrar su propia incertidumbre: asume carga operativa de forma automatizada (casi el 9% del volumen diario del BPO) incurriendo en un riesgo de clasificación errónea totalmente marginal y controlable, preservando el SLA del cliente. Esto justifica íntegramente la adopción del Deep Learning en este entorno industrial.

In [16]:
# CELDA 6: Entrenamiento Maestro y Auditoría Hold-Out Ciego
import os
import gc
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from scipy.special import softmax

print("\n=======================================================")
print("FASE FINAL: ENTRENAMIENTO MAESTRO Y AUDITORÍA HOLD-OUT")
print("=======================================================\n")

print("Ingestando matriz de entrenamiento Full-Shot...")
df_train_full = pd.read_parquet(DATA_PATH, engine='fastparquet')
df_train_full['full_text'] = df_train_full['full_text'].fillna("").astype(str)

print("Generando variables objetivo y tensor suavizado maestro...")
df_train_full['label'] = le.transform(df_train_full['target_tripleta'])

clases_unicas_full, frecuencias_full = np.unique(df_train_full['label'], return_counts=True)
pesos_crudos_full = 1.0 / np.sqrt(frecuencias_full)
pesos_numpy_full = pesos_crudos_full * (len(clases_unicas_full) / np.sum(pesos_crudos_full))
tensor_pesos_maestro = torch.tensor(pesos_numpy_full, dtype=torch.float32)

print("Transicionando a Apache Arrow y tokenizando en frío...")
dataset_train_full = Dataset.from_pandas(df_train_full[['full_text', 'label']])
del df_train_full
gc.collect()

dataset_train_tokenizado = dataset_train_full.map(tokenizar_lote, batched=True, batch_size=1000)
dataset_train_tokenizado = dataset_train_tokenizado.remove_columns(['full_text'])
del dataset_train_full
gc.collect()

print("\n--- Compilando Artefacto de Producción (RoBERTa 125M) ---")
modelo_maestro = AutoModelForSequenceClassification.from_pretrained(
    modelo_id,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

args_maestro = TrainingArguments(
    output_dir="/content/tmp_maestro",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=True,
    num_train_epochs=10,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    learning_rate=1e-4,
    warmup_steps=300,
    weight_decay=0.01,
    max_grad_norm=1.0
)

trainer_maestro = BPOWeightedTrainer(
    class_weights=tensor_pesos_maestro,
    model=modelo_maestro,
    args=args_maestro,
    train_dataset=dataset_train_tokenizado
)

print("Optimizando tensores masivos en CUDA...")
trainer_maestro.train()

print("\n=======================================================")
print("INFERENCIA SOBRE MATRIZ HOLD-OUT CIEGA")
print("=======================================================\n")

# ATENCIÓN: Esta ruta asume que subiste el test_set_telco.parquet a Colab.
# Si el nombre de tu Dataset en Colab es diferente, cambia DATA_PATH_TEST.
DATA_PATH_TEST = f'{BASE_PATH}/data/gold/test_set_telco.parquet'

try:
    print("Ingestando Hold-Out Test Set...")
    df_test = pd.read_parquet(DATA_PATH_TEST, engine='fastparquet')
    df_test['full_text'] = df_test['full_text'].fillna("").astype(str)
    df_test['label'] = le.transform(df_test['target_tripleta'])

    dataset_test = Dataset.from_pandas(df_test[['full_text', 'label']])
    dataset_test_tokenizado = dataset_test.map(tokenizar_lote, batched=True, batch_size=1000)
    dataset_test_tokenizado = dataset_test_tokenizado.remove_columns(['full_text'])

    print("Infiriendo red neuronal sobre matriz ciega...")
    predicciones_test = trainer_maestro.predict(dataset_test_tokenizado)
    logits_test = predicciones_test.predictions

    if isinstance(logits_test, tuple):
        logits_test = logits_test[0]

    if torch.is_tensor(logits_test):
        logits_test = logits_test.cpu().numpy()

    print("Normalizando distribuciones (Softmax)...")
    y_prob_test = softmax(logits_test, axis=1)
    y_true_test = np.array(dataset_test_tokenizado['label'])

    print("Calculando Telemetría Final BPO...\n")
    metricas_produccion = calcular_metricas_bpo(y_true_test, y_prob_test)

    print("--- VEREDICTO DE NEGOCIO ---")
    print(f"F1-Macro:               {metricas_produccion['f1_macro']:.4f}")
    print(f"F1-Weighted:            {metricas_produccion['f1_weighted']:.4f}")
    print(f"Log-Loss:               {metricas_produccion['log_loss']:.4f}")
    print(f"Tasa de Automatización: {metricas_produccion['tasa_automatizacion']:.2%}")
    print(f"Precisión Condicionada: {metricas_produccion['precision_condicionada']:.2%}\n")

    # Inyección de persistencia para el Hold-Out ciego
    df_metricas_produccion = pd.DataFrame([metricas_produccion])
    CSV_PRODUCCION_PATH = f"{BASE_PATH}/data/resultados_holdout_maestro_telco.csv"
    df_metricas_produccion.to_csv(CSV_PRODUCCION_PATH, index=False)
    print(f"Métricas maestras de producción exportadas a: {CSV_PRODUCCION_PATH}\n")

except Exception as e:
    print(f"ERROR CRÍTICO LEYENDO EL TEST SET: {e}\n¡Asegúrate de que el path DATA_PATH_TEST es correcto en Colab!")

print("Persistiendo orquestador y artefactos a disco...")
DIR_PRODUCCION = f"{BASE_PATH}/modelo/produccion_roberta"
trainer_maestro.save_model(DIR_PRODUCCION)
tokenizer.save_pretrained(DIR_PRODUCCION)
print(f"\nArquitectura sellada con éxito en: {DIR_PRODUCCION}")


FASE FINAL: ENTRENAMIENTO MAESTRO Y AUDITORÍA HOLD-OUT

Ingestando matriz de entrenamiento Full-Shot...
Generando variables objetivo y tensor suavizado maestro...
Transicionando a Apache Arrow y tokenizando en frío...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Map:   0%|          | 0/12322 [00:00<?, ? examples/s]


--- Compilando Artefacto de Producción (RoBERTa 125M) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Normalizando distribuciones (Softmax)...
Calculando Telemetría Final BPO...

--- VEREDICTO DE NEGOCIO ---
F1-Macro:               0.5112
F1-Weighted:            0.4974
Log-Loss:               1.7483
Tasa de Automatización: 8.70%
Precisión Condicionada: 89.55%



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Métricas maestras de producción exportadas a: /content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR/data/resultados_holdout_maestro_telco.csv

Persistiendo orquestador y artefactos a disco...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Arquitectura sellada con éxito en: /content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR/modelo/produccion_roberta
